# RSX-E05 | Study Feasibility & Sensitivity Analysis

Pre-registered sensitivity/feasibility calculations, not fabricated repair outcomes. Calculates rates, cohort deficits and projected sampling cost at observed yield. CPU-only, using original 272 records.

In [ ]:
import base64,io,zipfile,json,hashlib,collections,statistics,csv,pathlib,re,random,time
OUT=pathlib.Path('/content/RSX_EXPERIMENTS') if pathlib.Path('/content').exists() else pathlib.Path.cwd()/'RSX_EXPERIMENTS'
OUT.mkdir(exist_ok=True,parents=True)
def load_snapshot(b64):
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(b64))) as z:
        return ({n:json.loads(z.read(n)) for n in z.namelist() if n.endswith('.json')},z.namelist())
def write_json(path,x):
    path.parent.mkdir(exist_ok=True,parents=True)
    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(x,indent=2,default=str));tmp.replace(path)
def save_csv(path,rows):
    if not rows:return
    with path.open('w',newline='',encoding='utf8') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0].keys()));w.writeheader();w.writerows(rows)


In [ ]:
BASELINE_B64 = '...'

In [ ]:

data,_=load_snapshot(BASELINE_B64);rs=[v for k,v in data.items() if k.startswith('records/natural/')]
assert len(rs)==272
out=OUT/'RSX_E05';out.mkdir(exist_ok=True)
rows=[]
for split,need in [('calibration',4),('heldout',8)]:
 for kind in ['countdown','arithmetic','multistep']:
  a=[r for r in rs if r['split']==split and r['kind']==kind]
  success=sum(bool(r['evaluation']['eligible_semantic_failure']) for r in a)
  observed=success/max(len(a),1)
  remaining=max(need-success,0)
  # Planning-only estimates, not predictions.
  projected=round(remaining/observed) if observed>0 else None
  rows.append({'split':split,'kind':kind,'attempted':len(a),'eligible':success,
               'eligible_rate':round(observed,4),'required':need,'missing':remaining,
               'additional_if_rate_persists':projected})
summary={'original_trials':272,'eligible_total':sum(r['evaluation']['eligible_semantic_failure'] for r in rs),
 'missing_eligible':sum(r['missing'] for r in rows),
 'interpretation':'The sampling projections are descriptive and not guarantees. Zero yield has undefined projection.'}
save_csv(out/'cohort_deficits.csv',rows);write_json(out/'summary.json',summary)
print(json.dumps(summary,indent=2));print(*rows,sep=' | ');print('Saved:',out)
